# Lab 2 - Text Preprocessing and Regular Expressions

Completed solutions to the four tasks in the supplied Lab 2 notebook. The exercises use the specified Apple tweets dataset, Python regular expressions, spaCy, and NLTK.

Run the setup commands in the repository README first. All results below are produced by the code cells.

In [1]:
from pathlib import Path
import re
import pandas as pd
import nltk

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "download_data.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Run from the repository or one of its lab folders.")
nltk.data.path.insert(0, str(ROOT / ".nltk_data"))

## Task 1 - Count hashtags and find the top 10

Source: [Apple Twitter Sentiment Texts](https://www.kaggle.com/datasets/seriousran/appletwittersentimenttexts), uploaded by seriousran; the source page credits CrowdFlower.

We count every hashtag occurrence, including repeated hashtags in one tweet. Tags are converted to lowercase so `#Apple` and `#apple` are counted together. The regular expression requires at least one letter or underscore, excluding number-only fragments such as `#123`. It supports Unicode word characters. URLs are removed first to avoid counting URL fragments as hashtags. No stopword removal or stemming is used because it would change hashtag identities.

In [2]:
from collections import Counter
path = ROOT / "Lab 2/data/apple-twitter-sentiment-texts.csv"
if not path.is_file():
    raise FileNotFoundError("Run python download_data.py from the repository first.")
tweets = pd.read_csv(path)
print("Rows:", len(tweets))
print("Missing text:", tweets["text"].isna().sum())
tweets.head()

Rows: 1630
Missing text: 0


,text,sentiment
0,Wow. Yall needa step it up @Apple RT @heynyla:...,-1
1,What Happened To Apple Inc? http://t.co/FJEX...,0
2,Thank u @apple I can now compile all of the pi...,1
3,The oddly uplifting story of the Apple co-foun...,0
4,@apple can i exchange my iphone for a differen...,0


In [3]:
hashtag_pattern = re.compile(r"(?<!\w)#(?=\w*[^\W\d])\w+", flags=re.UNICODE)
def extract_hashtags(text):
    text = re.sub(r"https?://\S+|www\.\S+", " ", str(text))
    return [tag.lower() for tag in hashtag_pattern.findall(text)]

hashtags_per_tweet = tweets["text"].fillna("").map(extract_hashtags)
hashtag_counts = Counter(tag for tags in hashtags_per_tweet for tag in tags)
print("Total hashtag occurrences:", sum(hashtag_counts.values()))
print("Distinct hashtags:", len(hashtag_counts))
print("Tweets containing hashtags:", hashtags_per_tweet.map(bool).sum())
top_10 = pd.DataFrame(sorted(hashtag_counts.items(), key=lambda x: (-x[1], x[0]))[:10],
                      columns=["hashtag", "count"])
top_10

Total hashtag occurrences: 1604
Distinct hashtags: 550
Tweets containing hashtags: 783


,hashtag,count
0,#aapl,400
1,#apple,162
2,#iphone,38
3,#december,37
4,#iphone6,29
5,#stocks,15
6,#ipad,13
7,#iphone6plus,13
8,#iphone6s,13
9,#tech,12


## Task 2 - Use `re.compile()`

Compile a pattern that matches one or more digits, print its type, and replace the year. `re.compile()` creates a reusable pattern object with methods such as `search()`, `findall()`, and `sub()`. It keeps the pattern and any flags in one place when the same expression is used repeatedly.

In [4]:
text = "This year is 2021"
pattern = re.compile(r"\d+")
print("Pattern type:", type(pattern))
updated_text = pattern.sub("2022", text)
print(updated_text)

Pattern type: <class 're.Pattern'>
This year is 2022


## Task 3 - Use `re.split()`

`re.split()` separates a string wherever its pattern matches. Here, runs of digits act as separators. Spaces surrounding them remain in the output because the pattern matches digits only.

In [5]:
text = "a 11 b 2 3 c 4"
parts = re.split(r"\d+", text)
print(parts)

['a ', ' b ', ' ', ' c ', '']


## Task 4 - Compare spaCy and NLTK tokenization

`spacy.load("en_core_web_sm")` loads the installed English pipeline, including its tokenizer and trained language components. Calling the pipeline processes text into a `Doc`, which can be iterated to access individual tokens.

In [6]:
import spacy
nlp = spacy.load("en_core_web_sm")
text = "I'm enjoying the NLP course!"
spacy_tokens = [token.text for token in nlp(text)]
print("spaCy tokens:")
for token in spacy_tokens:
    print(token)
nltk_tokens = nltk.word_tokenize(text)
print("NLTK tokens:", nltk_tokens)
print("Same token sequence:", spacy_tokens == nltk_tokens)

spaCy tokens:
I
'm
enjoying
the
NLP
course
!
NLTK tokens: ['I', "'m", 'enjoying', 'the', 'NLP', 'course', '!']
Same token sequence: True


### Observation

For this sentence, both tokenizers produce `I`, `'m`, `enjoying`, `the`, `NLP`, `course`, and `!`. Both split the contraction and separate the punctuation. Matching on this example does not imply that their rules are identical for all text.

### Conclusion

The four tasks demonstrate hashtag extraction and aggregation, compiled patterns, regex splitting, and tokenization. Preprocessing depends on the task: preserving a hashtag is useful for counting tags, while separating punctuation is useful for word tokenization.

The dataset contains 1,630 tweets. With the stated matching rules, there are **1,604 hashtag occurrences**, **550 distinct hashtags**, and **783 tweets containing hashtags**. The most frequent tag is **#aapl (400 occurrences)**, followed by **#apple (162)**.